<a href="https://colab.research.google.com/github/25wh1a0529/Nasscom-lab/blob/main/U8_Data_Cleaning_and_Preprocessing_Part_1_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
print('Setup complete. pandas', pd.__version__)
raw = pd.DataFrame({
    'id':    [1, 2, 3, 4, 5, 6, 7, 7],
    'name':  ['Ana', 'Bo', 'Cy', 'Di', 'Eve', 'Fin', 'Gus', 'Gus'],
    'age':   [30, 25, np.nan, 41, -1, 38, 29, 29],
    'city':  [' Pune ', 'pune', 'DELHI', 'Delhi ', 'Mumbai', 'bombay', 'Pune.', 'Pune.'],
    'spend': ['120.5', '80.0', '200.2', 'N/A', '150.0', '99000', '110.0', '110.0'],
    'date':  ['2024-01-05', '2024-01-06', '2024-01-07', '2024-01-08',
              '2024-01-09', '2024-01-10', '2024-01-11', '2024-01-11'],
})
raw

Setup complete. pandas 2.2.3


,id,name,age,city,spend,date
0,1,Ana,30.0,Pune,120.5,2024-01-05
1,2,Bo,25.0,pune,80.0,2024-01-06
2,3,Cy,NaN,DELHI,200.2,2024-01-07
3,4,Di,41.0,Delhi,N/A,2024-01-08
4,5,Eve,-1.0,Mumbai,150.0,2024-01-09
5,6,Fin,38.0,bombay,99000,2024-01-10
6,7,Gus,29.0,Pune.,110.0,2024-01-11
7,7,Gus,29.0,Pune.,110.0,2024-01-11


In [ ]:
#1. Profile the data — find the problems
#  1A. A FEW COMMANDS REVEAL MOST PROBLEMS
df = raw.copy()
df.info()
# 1B. MISSING COUNTS, DUPLICATES & RANGES
print('Missing per column:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nNote: spend is type', df['spend'].dtype, "-> stored as text!")



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      8 non-null      int64  
 1   name    8 non-null      object 
 2   age     7 non-null      float64
 3   city    8 non-null      object 
 4   spend   8 non-null      object 
 5   date    8 non-null      object 
dtypes: float64(1), int64(1), object(4)
memory usage: 516.0+ bytes
Missing per column:
id       0
name     0
age      1
city     0
spend    0
date     0
dtype: int64

Duplicate rows: 1

Note: spend is type object -> stored as text!


In [ ]:
#LAB EXERCISE 1 :
# 1. duplicate row count
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

# 2. missing per column
missing_per_column = df.isnull().sum()
print("Missing values per column:")
print(missing_per_column)

# 3. Problems I can see: ...
# - Duplicate rows
# - Missing values
# - Inconsistent data types
# - Possible outliers
# - Possible inconsistent or invalid values

Duplicate rows: 1
Missing values per column:
id       0
name     0
age      1
city     0
spend    0
date     0
dtype: int64


In [ ]:
#2. Missing values — detect & handle
#  2A. UNMASK DISGUISED MISSING VALUES
df['spend'] = pd.to_numeric(df['spend'], errors='coerce')
df['age']   = df['age'].replace(-1, np.nan)

print('Missing after unmasking:')
print(df[['age', 'spend']].isna().sum())

#  2B. HANDLE THE GAPS (impute)
df['age']   = df['age'].fillna(df['age'].median())
df['spend'] = df['spend'].fillna(df['spend'].median())
print('Missing after imputing:', df[['age', 'spend']].isna().sum().sum())




Missing after unmasking:
age      2
spend    1
dtype: int64
Missing after imputing: 0


In [ ]:
# LAB EXERCISE 2 — Compare drop vs impute
ex = raw.copy()

# 1. unmask missing values (spend -> numeric, age -1 -> NaN)
ex["spend"] = pd.to_numeric(ex["spend"], errors="coerce")
ex["age"] = ex["age"].replace(-1, np.nan)

# 2a. dropna version
dropna_df = ex.dropna()
print("Dropna rows:", len(dropna_df))

# 2b. median-impute version
imputed_df = ex.copy()
imputed_df["age"] = imputed_df["age"].fillna(imputed_df["age"].median())
imputed_df["spend"] = imputed_df["spend"].fillna(imputed_df["spend"].median())

# 3. compare row counts
print("Original rows:", len(ex))
print("Dropna rows:", len(dropna_df))
print("Median-imputed rows:", len(imputed_df))

Dropna rows: 5
Original rows: 8
Dropna rows: 5
Median-imputed rows: 8


In [ ]:

#  3A. DROP DUPLICATE ROWS


print('Before:', df.shape)
df = df.drop_duplicates()
print('After :', df.shape, '-> removed the repeated Gus row')



#  3B. FIX DATA TYPES
df['date'] = pd.to_datetime(df['date'])

df['city'] = df['city'].astype('string')
print(df.dtypes)

Before: (8, 6)
After : (7, 6) -> removed the repeated Gus row
id                int64
name             object
age             float64
city     string[python]
spend           float64
date     datetime64[ns]
dtype: object


In [ ]:
# LAB EXERCISE 3 — Dedupe & retype
ex = raw.copy()

# 1. fix types: spend -> numeric, date -> datetime
ex["spend"] = pd.to_numeric(ex["spend"], errors="coerce")
ex["date"] = pd.to_datetime(ex["date"], errors="coerce")

# 2. drop duplicates
ex = ex.drop_duplicates()

# 3. dtypes + shape
print(ex.dtypes)
print("Shape:", ex.shape)

id                int64
name             object
age             float64
city             object
spend           float64
date     datetime64[ns]
dtype: object
Shape: (7, 6)


In [ ]:

# 4A. THE IQR RULE
q1, q3 = df['spend'].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f'Q1={q1:.1f}  Q3={q3:.1f}  IQR={iqr:.1f}')
print(f'Normal range: {low:.1f} to {high:.1f}')

outliers = df[(df['spend'] < low) | (df['spend'] > high)]
print('\nOutlier rows:')
print(outliers[['name', 'spend']])


#  4B. ONE WAY TO TREAT THEM — CAP (winsorise)
df['spend_capped'] = df['spend'].clip(lower=low, upper=high)
print(df[['name', 'spend', 'spend_capped']])

Q1=115.2  Q3=175.1  IQR=59.8
Normal range: 25.5 to 264.9

Outlier rows:
  name    spend
5  Fin  99000.0
  name    spend  spend_capped
0  Ana    120.5       120.500
1   Bo     80.0        80.000
2   Cy    200.2       200.200
3   Di    120.5       120.500
4  Eve    150.0       150.000
5  Fin  99000.0       264.875
6  Gus    110.0       110.000


In [ ]:
#LAB EXERCISE 4 — Find the outliers
# 1. Q1, Q3, IQR for 'age'
Q1 = ex["age"].quantile(0.25)
Q3 = ex["age"].quantile(0.75)
IQR = Q3 - Q1

# 2. lower & upper bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

# 3. rows outside the bounds
outliers = ex[(ex["age"] < lower_bound) | (ex["age"] > upper_bound)]
print(outliers)

Q1: 26.0
Q3: 36.0
IQR: 10.0
Lower bound: 11.0
Upper bound: 51.0
   id name  age    city  spend       date
4   5  Eve -1.0  Mumbai  150.0 2024-01-09


In [17]:

# 5A. THE PROBLEM — ONE CITY, MANY SPELLINGS

print(df['city'].value_counts())
# 5B. STANDARDISE THE STRINGS
s = df['city'].astype('string')
s = s.str.strip()
s = s.str.lower()
s = s.str.replace('.', '', regex=False)
s = s.replace({'bombay': 'mumbai'})
df['city'] = s
print(df['city'].value_counts())

city
pune      3
delhi     2
mumbai    2
Name: count, dtype: Int64
city
pune      3
delhi     2
mumbai    2
Name: count, dtype: Int64


In [ ]:
#LAB EXERCISE 5 — Clean a messy column
messy = pd.Series(
    [' London ', 'london', 'LONDON', 'N.Y.', 'new york ', 'New York'],
    dtype='string'
)

# 1. strip + lower
cleaned = messy.str.strip().str.lower()

# 2. map 'n.y.' -> 'new york' (after lowering)
cleaned = cleaned.replace('n.y.', 'new york')

# 3. value_counts()
print(cleaned.value_counts())

london      3
new york    3
Name: count, dtype: Int64


In [ ]:
#The cleaned dataset
# After all the steps above, here's the cleaned frame
clean = df.drop(columns=['spend_capped'])
print('Final shape:', clean.shape)
print('Missing values:', int(clean.isna().sum().sum()))
print('Duplicates    :', int(clean.duplicated().sum()))
clean


Final shape: (7, 6)
Missing values: 0
Duplicates    : 0


,id,name,age,city,spend,date
0,1,Ana,30.0,pune,120.5,2024-01-05
1,2,Bo,25.0,pune,80.0,2024-01-06
2,3,Cy,29.5,delhi,200.2,2024-01-07
3,4,Di,41.0,delhi,120.5,2024-01-08
4,5,Eve,29.5,mumbai,150.0,2024-01-09
5,6,Fin,38.0,mumbai,99000.0,2024-01-10
6,7,Gus,29.0,pune,110.0,2024-01-11
